# Ultra-Simple Binary Strategy
This notebook implements a minimalist 3-parameter strategy:
1. **Bias**: Global drift.
2. **Above-One**: Binary flag if the last close is > 1.0.
3. **Sentiment**: The sign of the latest headline sentiment (-1, 0, +1).
4. **Optimization**: Maximize Sharpe Ratio via Ridge Regression.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import seaborn as sns
import random

# GLOBAL CONFIG
SEED = 42
FIXED_WEIGHTS = {} 
FIXED_BIAS = None 

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(SEED)
sns.set_theme(style="darkgrid")


## 1. Load Data

In [ ]:
# Prices
train_seen = pd.read_parquet('data/bars_seen_train.parquet')
train_unseen = pd.read_parquet('data/bars_unseen_train.parquet')
test_seen_public = pd.read_parquet('data/bars_seen_public_test.parquet')
test_seen_private = pd.read_parquet('data/bars_seen_private_test.parquet')
test_seen = pd.concat([test_seen_public, test_seen_private], ignore_index=True)

# Headlines
h_seen_train = pd.read_parquet('data/headlines_seen_train.parquet')
h_seen_pub = pd.read_parquet('data/headlines_seen_public_test.parquet')
h_seen_priv = pd.read_parquet('data/headlines_seen_private_test.parquet')
test_headlines = pd.concat([h_seen_pub, h_seen_priv], ignore_index=True)


## 2. Ultra-Simple Feature Engineering

In [ ]:
def get_sentiment(text):
    text = text.lower()
    pos_words = ['secure', 'contract', 'growth', 'profit', 'expansion', 'office', 'partnership', 'gain', 'increase', 'success']
    neg_words = ['loss', 'lawsuit', 'investigation', 'deficit', 'decline', 'closure', 'fine', 'drop', 'decrease', 'fail']
    score = 0
    for w in pos_words:
        if w in text: score += 1
    for w in neg_words:
        if w in text: score -= 1
    return score

def extract_features(price_df, head_df):
    # Part A: Price Binary
    last_close = price_df.groupby('session')['close'].last()
    feat_df = pd.DataFrame(index=last_close.index)
    
    # 1. Binary: Close > 1
    feat_df['is_above_one'] = (last_close > 1.0).astype(float)
    
    # Part B: Sentiment Sign
    head_df = head_df.sort_values(['session', 'bar_ix'])
    latest_heads = head_df.groupby('session').tail(1).copy()
    latest_heads['latest_sentiment'] = latest_heads['headline'].apply(get_sentiment)
    
    # 2. Sign: -1, 0, or 1
    raw_sentiment = latest_heads.set_index('session')['latest_sentiment']
    feat_df['sentiment_sgn'] = np.sign(raw_sentiment.reindex(feat_df.index, fill_value=0))
    
    return feat_df

X_train_raw = extract_features(train_seen, h_seen_train)
X_test_raw = extract_features(test_seen, test_headlines)
y_train = (train_unseen.groupby('session')['close'].last() / train_seen.groupby('session')['close'].last()) - 1.0

# Normalize for ADAM stability
X_train = (X_train_raw - X_train_raw.mean()) / X_train_raw.std()
X_test = (X_test_raw - X_train_raw.mean()) / X_test_raw.std()
X_train = X_train.loc[y_train.index]

feature_names = X_train.columns.tolist()
print(f"Features: {feature_names}")


## 3. Training Loop (Manual L2 + Constraints)

In [ ]:
def calc_sharpe(positions, returns):
    pnl = positions * returns
    return np.mean(pnl) / (np.std(pnl) + 1e-9) * 16

class LinearSharpeModel(nn.Module):
    def __init__(self, n_features, feature_names, fixed_weights_dict, fixed_bias_val):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)
        self.register_buffer('w_mask', torch.ones(1, n_features))
        with torch.no_grad():
            for i, name in enumerate(feature_names):
                if name in fixed_weights_dict:
                    self.linear.weight[0, i] = fixed_weights_dict[name]
                    self.w_mask[0, i] = 0.0
            if fixed_bias_val is not None:
                self.linear.bias.data.fill_(fixed_bias_val)
                    
    def forward(self, x):
        return self.linear(x).squeeze()

def run_cv_stats(X, y, weight_decay_val):
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    val_sharpes = []
    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        set_seed(SEED)
        X_pt, y_pt = torch.tensor(X_tr.values, dtype=torch.float32), torch.tensor(y_tr.values, dtype=torch.float32)
        model = LinearSharpeModel(X_tr.shape[1], feature_names, FIXED_WEIGHTS, FIXED_BIAS)
        optimizer = optim.Adam(model.parameters(), lr=0.1, weight_decay=0.0)
        mask = model.w_mask
        for _ in range(600):
            optimizer.zero_grad()
            positions = model(X_pt)
            loss = -(torch.mean(positions * y_pt) / (torch.std(positions * y_pt) + 1e-9))
            loss += weight_decay_val * torch.sum((model.linear.weight * mask)**2)
            if FIXED_BIAS is None:
                loss += weight_decay_val * torch.sum(model.linear.bias**2)
            loss.backward()
            model.linear.weight.grad *= mask
            if FIXED_BIAS is not None:
                model.linear.bias.grad.zero_()
            optimizer.step()
        with torch.no_grad():
            va_pos = model(torch.tensor(X_va.values, dtype=torch.float32)).numpy()
            val_sharpes.append(calc_sharpe(va_pos, y_va.values))
    return np.min(val_sharpes), np.mean(val_sharpes)

# Config Sweep
wd_values = [0.0, 0.01, 0.1, 1.0, 5.0, 10.0]
best_min_val, best_wd = -1e9, 0
for wd in wd_values:
    min_s, mean_s = run_cv_stats(X_train, y_train, wd)
    print(f"WD: {wd:5} | Min Sharpe: {min_s:.4f} | Mean: {mean_s:.4f}")
    if min_s > best_min_val:
        best_min_val, best_wd = min_s, wd
print(f"\nBest WD: {best_wd}")


## 4. Final Training and Results

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
test_preds_all = np.zeros((5, len(X_test)))
fold_weights = []
fold_biases = []

for fold, (tr_idx, _) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    set_seed(SEED + fold)
    model = LinearSharpeModel(X_tr.shape[1], feature_names, FIXED_WEIGHTS, FIXED_BIAS)
    optimizer = optim.Adam(model.parameters(), lr=0.1, weight_decay=0.0)
    X_tr_t, y_tr_t = torch.tensor(X_tr.values, dtype=torch.float32), torch.tensor(y_tr.values, dtype=torch.float32)
    for _ in range(1001):
        optimizer.zero_grad()
        loss = -(torch.mean(model(X_tr_t)*y_tr_t) / (torch.std(model(X_tr_t)*y_tr_t)+1e-9))
        loss += best_wd * torch.sum((model.linear.weight * model.w_mask)**2)
        if FIXED_BIAS is None:
            loss += best_wd * torch.sum(model.linear.bias**2)
        loss.backward()
        model.linear.weight.grad *= model.w_mask
        if FIXED_BIAS is not None:
            model.linear.bias.grad.zero_()
        optimizer.step()
    with torch.no_grad():
        test_preds_all[fold] = model(torch.tensor(X_test.values, dtype=torch.float32)).numpy()
        fold_weights.append(model.linear.weight.squeeze().numpy())
        fold_biases.append(model.linear.bias.item())

avg_weights = np.mean(fold_weights, axis=0)
avg_bias = np.mean(fold_biases)

print("\n" + "="*40)
print("AVERAGED PARAMETERS (Ultra-Simple)")
print("="*40)
for name, weight in zip(feature_names, avg_weights):
    status = "[FIXED]" if name in FIXED_WEIGHTS else "[FREE]"
    print(f"{name:<25} : {weight:8.4f} {status}")
status_b = "[FIXED]" if FIXED_BIAS is not None else "[FREE]"
print(f"{'Bias (Drift)':<25} : {avg_bias:8.4f} {status_b}")
print("="*40)

submission = pd.DataFrame({'session': X_test.index, 'target_position': np.mean(test_preds_all, axis=0) * 1000})
submission.to_csv('submission_ultrasimple.csv', index=False)
